# Chapter 15 — Information Geometry and the Natural Gradient

Companion notebook to Chapter 15 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Expectations are exact here: sums over a finite set of outcomes, or a
quadrature on a fine grid for Gaussian densities. Nothing is sampled, except
where a cell says so.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, jacfwd, hessian

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor

def fisher_finite(logrho, theta, n_out):
    # Fisher matrix of a model on {0, ..., n_out - 1}: sum_y rho(y) [dl_y]^T [dl_y]
    F = torch.zeros(theta.numel(), theta.numel())
    for y in range(n_out):
        s = jacrev(lambda th: logrho(th, y))(theta)
        F = F + torch.exp(logrho(theta, y)) * torch.outer(s, s)
    return F

def fisher_hess(logrho, theta, n_out):
    # the other formula: - sum_y rho(y) [D^2 l_y]
    return -sum(torch.exp(logrho(theta, y)) * hessian(lambda th: logrho(th, y))(theta) for y in range(n_out))

# Gaussian family: expectations by the trapezoid rule on a fine grid
def gauss_E(h, mu, sigma, n=40001, width=12.0):
    ys = torch.linspace(mu - width * sigma, mu + width * sigma, n)
    w = torch.exp(-(ys - mu) ** 2 / (2 * sigma ** 2)) / math.sqrt(2 * math.pi * sigma ** 2)
    return torch.trapezoid(w * h(ys), ys)

2.15.0+cu132


## 1. Scores and the Fisher matrix

Lemma 15.4 (the score has mean zero), Theorem 15.6 (two formulas for the
Fisher matrix and its change of chart) and Example 15.7, with
Exercises 15.2 and 15.3.

In [2]:
# Bernoulli, charts theta and logit z
bern = lambda th, y: torch.log(th[0]) if y == 0 else torch.log(1 - th[0])
bern_logit = lambda z, y: bern(torch.sigmoid(z), y)
th = T([0.3]); z = torch.log(th / (1 - th))
F_th, F_z = fisher_finite(bern, th, 2), fisher_finite(bern_logit, z, 2)
print("Bernoulli, theta = 0.3:  F(theta) =", round(F_th.item(), 6), "  F(logit) =", round(F_z.item(), 6))
assert abs(F_th.item() - 1 / (0.3 * 0.7)) < 1e-12 and abs(F_z.item() - 0.3 * 0.7) < 1e-12
J = 1 / (0.3 * 0.7)                                              # d z / d theta
assert abs(F_z.item() - F_th.item() / J ** 2) < 1e-12            # F' = J^-T F J^-1
score_mean = sum(torch.exp(bern(th, y)) * jacrev(lambda t: bern(t, y))(th) for y in range(2))
assert score_mean.abs().max() < 1e-12
assert torch.allclose(F_th, fisher_hess(bern, th, 2))

# categorical with n = 3, chart (p1, p2)  (Exercise 15.2)
def cat(th, y):
    p = torch.cat([th, (1 - th.sum()).reshape(1)])
    return torch.log(p[y])
p = T([0.5, 0.3, 0.2]); th = p[:2]
F = fisher_finite(cat, th, 3)
print("categorical (0.5, 0.3, 0.2):  F =", [[round(v, 4) for v in row] for row in F.tolist()])
assert torch.allclose(F, torch.diag(1 / p[:2]) + 1 / p[2]) and torch.allclose(F, fisher_hess(cat, th, 3))
v = T([0.4, -0.1]); w = T([-0.2, 0.5])
full = lambda u: torch.cat([u, -u.sum().reshape(1)])
assert abs((v @ F @ w).item() - (full(v) * full(w) / p).sum().item()) < 1e-12
# the Hessian of one log-likelihood is not the Fisher matrix, and it is not even negative semidefinite in general
print("eigenvalues of -[D^2 l_y] for y = 0, 1, 2:",
      [[round(e, 3) + 0.0 for e in torch.linalg.eigvalsh(-hessian(lambda t: cat(t, y))(th)).tolist()] for y in range(3)])

# Gaussian, chart (mu, sigma), and chart (mu, s) with s = ln sigma  (Exercise 15.3)
mu, sg = 1.0, 2.0
dmu = lambda y: (y - mu) / sg ** 2
dsg = lambda y: -1 / sg + (y - mu) ** 2 / sg ** 3
FG = T([[gauss_E(lambda y: dmu(y) ** 2, mu, sg), gauss_E(lambda y: dmu(y) * dsg(y), mu, sg)],
        [gauss_E(lambda y: dmu(y) * dsg(y), mu, sg), gauss_E(lambda y: dsg(y) ** 2, mu, sg)]])
print("Gaussian (1, 2):  F =", [[round(v, 6) + 0.0 for v in row] for row in FG.tolist()])
assert torch.allclose(FG, torch.diag(T([1.0, 2.0])) / sg ** 2, atol=1e-9)
assert abs(gauss_E(dmu, mu, sg)) < 1e-12 and abs(gauss_E(dsg, mu, sg)) < 1e-10          # scores of mean zero
Jc = torch.diag(T([1.0, 1 / sg])); Ji = torch.linalg.inv(Jc)
assert torch.allclose(Ji.T @ (torch.diag(T([1.0, 2.0])) / sg ** 2) @ Ji, torch.diag(T([1 / sg ** 2, 2.0])))

Bernoulli, theta = 0.3:  F(theta) = 4.761905   F(logit) = 0.21
categorical (0.5, 0.3, 0.2):  F = [[7.0, 5.0], [5.0, 8.3333]]
eigenvalues of -[D^2 l_y] for y = 0, 1, 2: [[0.0, 4.0], [0.0, 11.111], [0.0, 50.0]]
Gaussian (1, 2):  F = [[0.25, 0.0], [0.0, 0.5]]


## 2. The simplex as a piece of a sphere

Proposition 15.8, Figure 15.1 and Exercise 15.1.

In [3]:
S = lambda th: 2 * torch.sqrt(torch.cat([th, (1 - th.sum()).reshape(1)]))
JS = jacfwd(S)(th)
assert torch.allclose(JS.T @ JS, F)                              # the Fisher metric is the pull-back of the sphere
assert abs(S(th).norm().item() - 2) < 1e-12

def fisher_length(path, n=20001):                                # length of a curve of distributions
    ts = torch.linspace(0.0, 1.0, n)
    P = torch.stack([path(t) for t in ts])
    mid = 0.5 * (P[1:] + P[:-1]); d = P[1:] - P[:-1]
    return torch.sqrt((d ** 2 / mid).sum(1)).sum().item()
q = T([0.1, 0.2, 0.7])
dist = 2 * math.acos(torch.sqrt(p * q).sum().item())
sp, sq = torch.sqrt(p), torch.sqrt(q)
arc = lambda t: ((1 - t) * sp + t * sq) ** 2 / (((1 - t) * sp + t * sq) ** 2).sum()   # the great arc, in the simplex
segment = lambda t: (1 - t) * p + t * q
print("2 arccos sum sqrt(p q) =", round(dist, 6), "  length of the great arc:", round(fisher_length(arc), 6),
      "  of the straight segment:", round(fisher_length(segment), 6))
assert abs(fisher_length(arc) - dist) < 1e-6 and fisher_length(segment) > dist + 1e-4

# Bernoulli: equally spaced points on the quarter circle, total length pi, flat chart phi
alphas = torch.arange(1, 8) * math.pi / 16
print("Figure 15.1, theta of the seven points:", [round(v, 3) for v in (torch.sin(alphas) ** 2).tolist()])
phi = torch.linspace(1e-6, math.pi - 1e-6, 200001)
theta = torch.sin(phi / 2) ** 2
total = torch.trapezoid(torch.sqrt(1 / (theta * (1 - theta))) * 0.5 * torch.sin(phi), phi)   # sqrt(F) dtheta
print("total Fisher length of the Bernoulli family:", round(total.item(), 6))
assert abs(total.item() - math.pi) < 1e-5
bern_phi = lambda ph, y: bern(torch.sin(ph / 2) ** 2, y)
assert abs(fisher_finite(bern_phi, T([1.1]), 2).item() - 1) < 1e-12

2 arccos sum sqrt(p q) = 1.136955   length of the great arc: 1.136955   of the straight segment: 1.141804


Figure 15.1, theta of the seven points: [0.038, 0.146, 0.309, 0.5, 0.691, 0.854, 0.962]
total Fisher length of the Bernoulli family: 3.141591


## 3. The Kullback–Leibler divergence

Proposition 15.11, Theorem 15.12 and Exercise 15.4.

In [4]:
kl = lambda a, b: (a * torch.log(a / b)).sum()
torch.manual_seed(1)
for _ in range(1000):
    a, b = torch.softmax(torch.randn(4), 0), torch.softmax(torch.randn(4), 0)
    assert kl(a, b) >= 0
assert kl(p, p) == 0
b2 = lambda t: T([t, 1 - t])
print("D(0.5 || 0.1) =", round(kl(b2(0.5), b2(0.1)).item(), 4), "   D(0.1 || 0.5) =", round(kl(b2(0.1), b2(0.5)).item(), 4))
assert abs(kl(b2(0.5), b2(0.1)).item() - 0.5108) < 1e-4 and abs(kl(b2(0.1), b2(0.5)).item() - 0.3681) < 1e-4
for h in (0.1, 0.01):
    print(f"h = {h}:  D(0.5 || 0.5 + h) = {kl(b2(0.5), b2(0.5 + h)).item():.8f}   (1/2) F h^2 = {2 * h * h:.8f}")
assert abs(kl(b2(0.5), b2(0.6)).item() - 0.02041) < 1e-5 and abs(kl(b2(0.5), b2(0.51)).item() - 0.00020004) < 1e-8

# the Hessian of both divergences at q = p is the Fisher matrix (categorical, chart (p1, p2))
probs = lambda t: torch.cat([t, (1 - t.sum()).reshape(1)])
H1 = hessian(lambda t: kl(probs(th), probs(t)))(th)
H2 = hessian(lambda t: kl(probs(t), probs(th)))(th)
assert torch.allclose(H1, F) and torch.allclose(H2, F)
assert jacrev(lambda t: kl(probs(th), probs(t)))(th).abs().max() < 1e-12
print("Hessian of q -> D(p || q) and of q -> D(q || p) at q = p: the Fisher matrix")

D(0.5 || 0.1) = 0.5108    D(0.1 || 0.5) = 0.3681
h = 0.1:  D(0.5 || 0.5 + h) = 0.02041100   (1/2) F h^2 = 0.02000000
h = 0.01:  D(0.5 || 0.5 + h) = 0.00020004   (1/2) F h^2 = 0.00020000
Hessian of q -> D(p || q) and of q -> D(q || p) at q = p: the Fisher matrix


## 4. The natural gradient and its invariance

Proposition 15.14, Theorem 15.15, Figure 15.2 and Exercise 15.6, on the
Gaussian family with the loss of Chapter 13.

In [5]:
L = lambda ms: torch.log(ms[1]) + (1 + ms[0] ** 2) / (2 * ms[1] ** 2)       # in the chart (mu, sigma)
Fms = lambda ms: torch.diag(torch.stack([1 / ms[1] ** 2, 2 / ms[1] ** 2]))
charts = {
    "(mu, sigma)":    (lambda x: x,                                    lambda ms: ms),
    "(mu, ln sigma)": (lambda x: torch.stack([x[0], torch.exp(x[1])]), lambda ms: torch.stack([ms[0], torch.log(ms[1])])),
    "(mu, sigma^2)":  (lambda x: torch.stack([x[0], torch.sqrt(x[1])]), lambda ms: torch.stack([ms[0], ms[1] ** 2])),
}
ms0 = T([1.0, 2.0])
def one_step(name, eta, natural):
    to_ms, from_ms = charts[name]
    x0 = from_ms(ms0)
    row = jacrev(lambda x: L(to_ms(x)))(x0)
    if natural:
        Ji = torch.linalg.inv(jacrev(from_ms)(ms0))
        Fx = Ji.T @ Fms(ms0) @ Ji                                 # the Fisher matrix in this chart
        row = torch.linalg.solve(Fx, row)
    return to_ms(x0 - eta * row), row
for natural in (False, True):
    ends = {name: one_step(name, 0.4, natural)[0] for name in charts}
    print("natural gradient" if natural else "gradient descent", "with eta = 0.4:",
          {k: [round(a, 4) for a in v.tolist()] for k, v in ends.items()})
nat = {name: one_step(name, 0.4, True)[0] for name in charts}
assert torch.allclose(nat["(mu, sigma)"], T([0.6, 1.8])) and torch.allclose(nat["(mu, ln sigma)"], T([0.6, 2 * math.exp(-0.1)]))
assert torch.allclose(nat["(mu, sigma^2)"], T([0.6, math.sqrt(3.2)]))
gd = {name: one_step(name, 0.4, False)[0] for name in charts}
assert torch.allclose(gd["(mu, ln sigma)"], T([0.9, 2 * math.exp(-0.2)])) and torch.allclose(gd["(mu, sigma^2)"], T([0.9, math.sqrt(3.975)]))
# the natural gradient is one tangent vector: components (1, 1/2) and (1, 1/4), related by J
g1, g2 = one_step("(mu, sigma)", 0.4, True)[1], one_step("(mu, ln sigma)", 0.4, True)[1]
assert torch.allclose(g1, T([1.0, 0.5])) and torch.allclose(g2, T([1.0, 0.25])) and torch.allclose(g2, torch.diag(T([1.0, 0.5])) @ g1)
# order of the disagreement between two charts: eta^2 for the natural gradient, eta for gradient descent
for natural in (True, False):
    gaps = [(one_step("(mu, sigma)", e, natural)[0] - one_step("(mu, ln sigma)", e, natural)[0]).norm().item() for e in (0.1, 0.01)]
    print("natural" if natural else "plain  ", "gap between the two charts for eta = 0.1, 0.01:", [f"{g:.2e}" for g in gaps],
          " ratio", round(gaps[0] / gaps[1], 1))
    assert (80 < gaps[0] / gaps[1] < 125) if natural else (8 < gaps[0] / gaps[1] < 12.5)
# one step with eta = 1 in the chart (mu, sigma) finds the mean
for start in (T([1.0, 2.0]), T([-3.0, 0.7])):
    assert abs((start - torch.linalg.solve(Fms(start), jacrev(L)(start)))[0].item()) < 1e-12

# Proposition 15.14: steepest descent on a Fisher sphere
row = jacrev(L)(ms0); Fm = Fms(ms0); eps = 0.01
hstar = -math.sqrt(2 * eps / (row @ torch.linalg.solve(Fm, row)).item()) * torch.linalg.solve(Fm, row)
assert abs(0.5 * (hstar @ Fm @ hstar).item() - eps) < 1e-12
torch.manual_seed(2)
for _ in range(2000):
    h = torch.randn(2); h = h * math.sqrt(2 * eps / (h @ Fm @ h).item())
    assert row @ h >= row @ hstar - 1e-12
print("Proposition 15.14: no direction on the Fisher sphere decreases the linearized loss more")

gradient descent with eta = 0.4: {'(mu, sigma)': [0.9, 1.9], '(mu, ln sigma)': [0.9, 1.6375], '(mu, sigma^2)': [0.9, 1.9937]}
natural gradient with eta = 0.4: {'(mu, sigma)': [0.6, 1.8], '(mu, ln sigma)': [0.6, 1.8097], '(mu, sigma^2)': [0.6, 1.7889]}
natural gap between the two charts for eta = 0.1, 0.01: ['6.20e-04', '6.24e-06']  ratio 99.3
plain   gap between the two charts for eta = 0.1, 0.01: ['7.25e-02', '7.48e-03']  ratio 9.7
Proposition 15.14: no direction on the Fisher sphere decreases the linearized loss more


## 5. Exponential families

Theorem 15.17 on the categorical family in logits (the last one fixed to 0),
Example 15.18 and Exercise 15.5 (Poisson).

In [6]:
A = lambda t: torch.logsumexp(torch.cat([t, torch.zeros(1)]), 0)             # log-partition; T(y) = e_y for y < n, 0 for y = n
soft = lambda t: torch.softmax(torch.cat([t, torch.zeros(1)]), 0)
t0 = T([0.4, -0.7, 1.1]); pr = soft(t0); mu = pr[:3]
catl = lambda t, y: torch.log(soft(t)[y])
Fe = fisher_finite(catl, t0, 4)
assert torch.allclose(jacrev(A)(t0), mu)                                      # dA = mean parameters
assert torch.allclose(hessian(A)(t0), Fe)                                     # D^2 A = Fisher matrix
Tm = torch.cat([torch.eye(3), torch.zeros(1, 3)])                             # rows: T(y)
cov = sum(pr[y] * torch.outer(Tm[y] - mu, Tm[y] - mu) for y in range(4))
assert torch.allclose(cov, Fe) and torch.allclose(Fe, torch.diag(mu) - torch.outer(mu, mu))
t1 = T([-0.2, 0.3, 0.5])
assert abs(kl(soft(t0), soft(t1)).item() - (A(t1) - A(t0) - jacrev(A)(t0) @ (t1 - t0)).item()) < 1e-12
# mean chart: Jacobian F, Fisher matrix F^{-1}, and F^{-1} [d_theta L]^T = [d_mu L]^T
Jm = jacrev(lambda t: soft(t)[:3])(t0)
assert torch.allclose(Jm, Fe)
cat_mu = lambda m, y: torch.log(torch.cat([m, (1 - m.sum()).reshape(1)])[y])
assert torch.allclose(fisher_finite(cat_mu, mu, 4), torch.linalg.inv(Fe))
cvec = T([0.3, -1.0, 2.0, 0.5])
Lm = lambda m: (torch.cat([m, (1 - m.sum()).reshape(1)]) ** 2 * cvec).sum()   # some function of the distribution
Lt = lambda t: Lm(soft(t)[:3])
assert torch.allclose(torch.linalg.solve(Fe, jacrev(Lt)(t0)), jacrev(Lm)(mu))
print("categorical family: dA = mu, D^2 A = F = Cov(T), KL = Bregman divergence of A, F^-1 [d_theta L]^T = [d_mu L]^T")

# Bernoulli and fixed-variance Gaussian
Ab = lambda z: torch.log(1 + torch.exp(z)); zb = T(0.8); mb = torch.sigmoid(zb)
assert abs(jacfwd(jacfwd(Ab))(zb).item() - (mb * (1 - mb)).item()) < 1e-12
# Poisson (Exercise 15.5), with the series truncated far in the tail
lam_, lam2 = 3.0, 4.5
ys = torch.arange(0, 80.0)
pois = lambda lam: torch.exp(-lam + ys * math.log(lam) - torch.lgamma(ys + 1))
assert abs(pois(lam_).sum().item() - 1) < 1e-12
score_theta = ys - lam_                                                       # d/dtheta of y theta - e^theta
print("Poisson, lambda = 3:  F(theta) =", round((pois(lam_) * score_theta ** 2).sum().item(), 6),
      "  F(lambda) =", round((pois(lam_) * (ys / lam_ - 1) ** 2).sum().item(), 6))
assert abs((pois(lam_) * score_theta ** 2).sum().item() - lam_) < 1e-9
assert abs((pois(lam_) * (ys / lam_ - 1) ** 2).sum().item() - 1 / lam_) < 1e-9
klp = (pois(lam_) * (torch.log(pois(lam_)) - torch.log(pois(lam2)))).sum().item()
assert abs(klp - (lam2 - lam_ - lam_ * math.log(lam2 / lam_))) < 1e-9

categorical family: dA = mu, D^2 A = F = Cov(T), KL = Bregman divergence of A, F^-1 [d_theta L]^T = [d_mu L]^T
Poisson, lambda = 3:  F(theta) = 3.0   F(lambda) = 0.333333


## 6. Models with inputs: Fisher, Gauss–Newton, Hessian

Proposition 15.19 on a small network with softmax outputs, the pitfall on
the empirical Fisher matrix, and Exercise 15.8.

In [7]:
torch.manual_seed(3)
N, din, dh, K = 12, 3, 4, 3
X = torch.randn(N, din); Y = torch.randint(0, K, (N,))
sizes = [(dh, din), (dh,), (K, dh)]
def unpack(th):
    out, i = [], 0
    for shp in sizes:
        n = math.prod(shp); out.append(th[i:i + n].reshape(shp)); i += n
    return out
def f(x, th):                                                     # natural parameters (logits) of the label
    W1, b1, W2 = unpack(th)
    return W2 @ torch.tanh(W1 @ x + b1)
d = sum(math.prod(s) for s in sizes)
th = 0.5 * torch.randn(d)
logp = lambda th, x, y: torch.log_softmax(f(x, th), 0)[y]
loss = lambda th: -sum(logp(th, X[n], Y[n]) for n in range(N)) / N
AK = lambda z: torch.logsumexp(z, 0)
Fnet = torch.zeros(d, d); Fdef = torch.zeros(d, d); corr = torch.zeros(d, d)
for n in range(N):
    z = f(X[n], th); Jn = jacrev(lambda t: f(X[n], t))(th)
    Fnet += Jn.T @ hessian(AK)(z) @ Jn / N
    pz = torch.softmax(z, 0)
    for y in range(K):                                            # the definition: labels drawn from the model
        s = jacrev(lambda t: logp(t, X[n], y))(th)
        Fdef += pz[y] * torch.outer(s, s) / N
    resid = torch.nn.functional.one_hot(Y[n], K) - pz             # T(y_n) - dA(z_n)
    Hf = hessian(lambda t: f(X[n], t))(th)                        # K x d x d
    corr += torch.einsum("k,kij->ij", resid, Hf) / N
Hloss = hessian(loss)(th)
assert torch.allclose(Fnet, Fdef, atol=1e-12) and torch.allclose(Hloss, Fnet - corr, atol=1e-10)
print("smallest eigenvalue of the Fisher / Gauss-Newton matrix:", f"{torch.linalg.eigvalsh(Fnet)[0].item():.1e}",
      "   of the Hessian of the loss:", round(torch.linalg.eigvalsh(Hloss)[0].item(), 4))
assert torch.linalg.eigvalsh(Fnet)[0] > -1e-12 and torch.linalg.eigvalsh(Hloss)[0] < -1e-3

# a model that is linear in theta (softmax regression): the Fisher matrix is the Hessian
flin = lambda x, W: W.reshape(K, din) @ x
Wl = torch.randn(K * din)
loss_lin = lambda W: -sum(torch.log_softmax(flin(X[n], W), 0)[Y[n]] for n in range(N)) / N
Flin = sum(jacrev(lambda W: flin(X[n], W))(Wl).T @ hessian(AK)(flin(X[n], Wl)) @ jacrev(lambda W: flin(X[n], W))(Wl) for n in range(N)) / N
assert torch.allclose(Flin, hessian(loss_lin)(Wl), atol=1e-12)

# Exercise 15.8: linear regression with unit noise
Xr = torch.randn(20, 3); theta_true = T([1.0, -2.0, 0.5]); yr = Xr @ theta_true      # fitted exactly
Lr = lambda t: 0.5 * ((Xr @ t - yr) ** 2).mean()
Fr = Xr.T @ Xr / 20
assert torch.allclose(hessian(Lr)(torch.zeros(3)), Fr)
start = torch.randn(3)
assert torch.allclose(start - torch.linalg.solve(Fr, jacrev(Lr)(start)), theta_true)  # one natural gradient step
emp = lambda t: sum((yr[n] - Xr[n] @ t) ** 2 * torch.outer(Xr[n], Xr[n]) for n in range(20)) / 20
print("at the exact fit: |empirical Fisher| =", emp(theta_true).norm().item(), "  |Fisher| =", round(Fr.norm().item(), 4))
assert emp(theta_true).norm() == 0 and Fr.norm() > 0.5

smallest eigenvalue of the Fisher / Gauss-Newton matrix: -1.7e-17    of the Hessian of the loss: -0.5154


at the exact fit: |empirical Fisher| = 0.0   |Fisher| = 1.9309


## 7. Mirror descent

Proposition 15.21, Example 15.22 with Exercise 15.7, and Theorem 15.23.

In [8]:
# Exercise 15.7: exponentiated gradient on the simplex
c = T([1.0, 2.0, 3.0]); eta = math.log(2)
pk = torch.ones(3) / 3
p_new = pk * torch.exp(-eta * c); p_new = p_new / p_new.sum()
print("exponentiated gradient step:", [round(v, 6) for v in p_new.tolist()], "= (4, 2, 1) / 7")
assert torch.allclose(p_new, T([4.0, 2.0, 1.0]) / 7)
assert torch.allclose(torch.softmax(torch.log(pk) - eta * c, 0), p_new)       # a step on the logits
# it minimizes dL(p_k)(p - p_k) + D(p || p_k) / eta over the simplex
obj = lambda q_: c @ (q_ - pk) + kl(q_, pk) / eta
torch.manual_seed(4)
assert all(obj(torch.softmax(torch.randn(3), 0)) >= obj(p_new) - 1e-12 for _ in range(3000))

# Theorem 15.23 with psi(a) = sum a ln a on positive vectors: chart m = 1 + ln a, metric G = diag(1/a)
Lq = lambda a: ((a - T([0.5, 1.5, 1.0])) ** 2).sum() + a[0] * a[1]
a0 = T([0.8, 0.4, 1.3]); eta = 0.2
m0 = 1 + torch.log(a0)
a_mirror = torch.exp(m0 - eta * jacrev(Lq)(a0) - 1)                           # d psi(a+) = d psi(a) - eta dL(a)
a_of_m = lambda m: torch.exp(m - 1)
Gm = torch.diag(a0)                                                           # G' = G^{-1} in the chart m
m_riem = m0 - eta * torch.linalg.solve(Gm, jacrev(lambda m: Lq(a_of_m(m)))(m0))
assert torch.allclose(a_of_m(m_riem), a_mirror)
print("mirror descent step = Riemannian gradient step in the chart m:", [round(v, 6) for v in a_mirror.tolist()])
assert (a_mirror > 0).all()

# exponential family: mirror descent with potential A = natural gradient descent in the mean chart, exactly
eta = 0.3
mu_mirror = jacrev(A)(t0) - eta * jacrev(Lt)(t0)                              # d A(theta+) = d A(theta) - eta dL
F_mu = torch.linalg.inv(Fe)                                                   # Fisher matrix in the mean chart
mu_ngd = mu - eta * torch.linalg.solve(F_mu, jacrev(Lm)(mu))
assert torch.allclose(mu_mirror, mu_ngd)
print("mirror descent with the log-partition function = natural gradient descent in the mean parameters")

exponentiated gradient step: [0.571429, 0.285714, 0.142857] = (4, 2, 1) / 7
mirror descent step = Riemannian gradient step in the chart m: [0.654985, 0.529252, 1.152997]
mirror descent with the log-partition function = natural gradient descent in the mean parameters


## 8. Machine learning: what the approximations keep

Table 15.1 and Exercise 15.9, the Kronecker-factored form of one layer,
Adam with steady gradients, and the trust-region step.

In [9]:
# invariances, checked on the Gaussian family: chart x = (mu, sigma), chart x' = B x (a linear change)
row = jacrev(L)(ms0); Fm = Fms(ms0); H = hessian(L)(ms0)
steps = {
    "gradient descent": lambda r, F_, H_: -0.1 * r,
    "sign descent":     lambda r, F_, H_: -0.1 * torch.sign(r),
    "diagonal Fisher":  lambda r, F_, H_: -0.1 * r / torch.diagonal(F_),
    "Newton":           lambda r, F_, H_: -torch.linalg.solve(H_, r),
    "natural gradient": lambda r, F_, H_: -0.1 * torch.linalg.solve(F_, r),
}
def same_point(rule, B, cL=1.0):
    Bi = torch.linalg.inv(B)
    step = rule(cL * row, Fm, cL * H)                                         # in the chart x
    step_p = rule(cL * row @ Bi, Bi.T @ Fm @ Bi, cL * Bi.T @ H @ Bi)          # in the chart x' = B x
    return torch.allclose(B @ step, step_p)
def loss_scaling(rule):
    return torch.allclose(rule(3.0 * row, Fm, 3.0 * H), rule(row, Fm, H))
Bdiag = torch.diag(T([2.0, 0.5])); Blin = T([[1.0, 0.7], [-0.4, 1.5]])
expected = {"gradient descent": (False, False, False), "sign descent": (True, False, False),
            "diagonal Fisher": (False, True, False), "Newton": (True, True, True), "natural gradient": (False, True, True)}
for name, rule in steps.items():
    got = (loss_scaling(rule), same_point(rule, Bdiag), same_point(rule, Blin))
    print(f"{name:17s} loss x c: {str(got[0]):5s}  rescale coordinates: {str(got[1]):5s}  linear change: {str(got[2]):5s}")
    assert got == expected[name]

# Adam with a constant gradient takes steps of size eta in every coordinate
w = torch.nn.Parameter(T([0.0, 0.0, 0.0])); opt = torch.optim.Adam([w], lr=0.01, eps=1e-12)
gconst = T([5.0, -0.02, 300.0])
for _ in range(5):
    opt.zero_grad(); w.grad = gconst.clone(); opt.step()
print("Adam, five steps with gradient (5, -0.02, 300):", [round(v, 6) for v in w.detach().tolist()])
assert torch.allclose(w.detach(), -0.05 * torch.sign(gconst), atol=1e-8)

# Kronecker-factored Fisher form of a layer s = W a
torch.manual_seed(5)
nin, nout = 3, 2
As = torch.randn(6, nin); Ds = torch.randn(5, nout)
pairs = [(a, dl) for a in As for dl in Ds]                                    # a and delta independent
Acal = As.T @ As / len(As); Gam = Ds.T @ Ds / len(Ds)
V, V2 = torch.randn(nout, nin), torch.randn(nout, nin)
exact = sum((dl @ V @ a) * (dl @ V2 @ a) for a, dl in pairs) / len(pairs)
assert abs(exact.item() - torch.trace(V.T @ Gam @ V2 @ Acal).item()) < 1e-12
grad_W = torch.randn(nout, nin)
Xnat = torch.linalg.solve(Gam, grad_W) @ torch.linalg.inv(Acal)
Ffull = sum(torch.outer(torch.outer(dl, a).reshape(-1), torch.outer(dl, a).reshape(-1)) for a, dl in pairs) / len(pairs)
assert torch.allclose(torch.linalg.solve(Ffull, grad_W.reshape(-1)).reshape(nout, nin), Xnat)
dep = [(As[i], Ds[i] * As[i, 0]) for i in range(5)]                           # dependent pairs: only an approximation
ex_dep = sum((dl @ V @ a) * (dl @ V2 @ a) for a, dl in dep) / 5
Ad = sum(torch.outer(a, a) for a, _ in dep) / 5; Gd = sum(torch.outer(dl, dl) for _, dl in dep) / 5
print("independent a and delta: exact.  Dependent:", round(ex_dep.item(), 4), "against", round(torch.trace(V.T @ Gd @ V2 @ Ad).item(), 4))
print("natural gradient of the layer = Gamma^-1 [dL/dW] A^-1: solved with a", nout, "x", nout, "and a", nin, "x", nin,
      "matrix instead of", nout * nin, "x", nout * nin)

# trust region: the step of Proposition 15.14 changes the policy by a divergence close to delta
policy = lambda t: torch.softmax(t, 0)
tpol = T([0.2, -0.5, 1.0, 0.1]); reward = T([1.0, 0.0, 2.0, -1.0])
Lpol = lambda t: -(policy(t) * reward).sum()
Fp = torch.diag(policy(tpol)) - torch.outer(policy(tpol), policy(tpol)) + 1e-9 * torch.eye(4)   # singular along (1,...,1)
gp = jacrev(Lpol)(tpol)
for delta in (1e-2, 1e-3):
    nat = torch.linalg.lstsq(Fp, gp.reshape(-1, 1)).solution.reshape(-1)
    h = -math.sqrt(2 * delta / (gp @ nat).item()) * nat
    real = kl(policy(tpol), policy(tpol + h)).item()
    hg = -gp * math.sqrt(2 * delta / (gp @ Fp @ gp).item())                   # plain gradient, same divergence budget
    print(f"delta = {delta}:  divergence of the natural step {real:.5f};  loss change {Lpol(tpol + h).item() - Lpol(tpol).item():+.5f}"
          f"  (plain gradient with the same budget: {Lpol(tpol + hg).item() - Lpol(tpol).item():+.5f})")
    assert abs(real / delta - 1) < 0.2 and gp @ h <= gp @ hg + 1e-12

gradient descent  loss x c: False  rescale coordinates: False  linear change: False
sign descent      loss x c: True   rescale coordinates: False  linear change: False
diagonal Fisher   loss x c: False  rescale coordinates: True   linear change: False
Newton            loss x c: True   rescale coordinates: True   linear change: True 
natural gradient  loss x c: False  rescale coordinates: True   linear change: True 
Adam, five steps with gradient (5, -0.02, 300): [-0.05, 0.05, -0.05]
independent a and delta: exact.  Dependent: 0.0597 against -0.0776
natural gradient of the layer = Gamma^-1 [dL/dW] A^-1: solved with a 2 x 2 and a 3 x 3 matrix instead of 6 x 6
delta = 0.01:  divergence of the natural step 0.00966;  loss change -0.15688  (plain gradient with the same budget: -0.15562)
delta = 0.001:  divergence of the natural step 0.00099;  loss change -0.05152  (plain gradient with the same budget: -0.05049)


## Summary

* The Fisher matrix is the second moment of the score and minus the mean of
  the second derivatives of the log-likelihood; it changes as
  `J^{-T} F J^{-1}`.
* The simplex with the Fisher metric is a piece of a sphere of radius 2; the
  Bernoulli family has length `pi`.
* The Hessian of the Kullback–Leibler divergence at `q = p` is the Fisher
  matrix, in both arguments.
* Natural gradient steps computed in different charts agree to second order
  in the step; gradient descent steps differ at first order.
* In an exponential family `F = D^2 A`, and the natural gradient in one
  chart is the row of partial derivatives in the dual chart; for models with
  inputs `F` is the Gauss–Newton part of the Hessian.
* Mirror descent with potential `psi` is a Riemannian gradient step for the
  metric `D^2 psi` in the chart `d psi`.